## Exercice n°2 (1/2 h)



[texte du lien](https://)Télécharger le Visual Transformer (DeiT) hébergé sur Hugging Face à l'adresse :
"facebook/deit-tiny-patch16-224". Répondez aux questions suivantes en vous appuyant sur des éléments de l'architecture ou d'une des fonctions associées au modèle (en particulier la fonction forward()).
- Quelle est la taille de l'espace latent de ce transformer ?
- Comment l'image est-elle tokenisée et en quoi consiste l'embedding (valeurs et position) ?
- Quelle est la signification du premier token ?
- Déterminer et interpréter l'output associé à l'image fournie dans le TP n°2 (cat.jpg) à l'aide de ce modèle.
- Visualiser les scores attentionnels associés au premier token pour les premières têtes de la première couche attentionnelle. Comparer à ce qu'on avait obtenu pour un ResNet50 pré-entraîné sur ImageNet.

In [2]:
import torch
from transformers import AutoImageProcessor, ViTForImageClassification
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Q0
model_name = "facebook/deit-tiny-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(model_name)
model.eval()  # mode evaluation
#Q1
#La taille de l’espace latent correspond à la dimension des embeddings internes du modèle, notée hidden_size.
#hidden_size = 192



#Q2 :

# Charger l'image
img = Image.open("../exam_session_anticipee/cat.jpg").convert("RGB")
plt.imshow(img)
plt.title("Image d'entrée")
plt.axis('off')
plt.show()
# ===============================
# 2️⃣ Tokenisation et embedding
# ===============================
inputs = processor(images=img, return_tensors="pt")
pixel_values = inputs["pixel_values"]
print("Pixel values shape (batch, channels, H, W):", pixel_values.shape)

# Le modèle convertit les patchs de 16x16 en tokens
print("Taille des tokens : chaque patch → vecteur de dimension 192 (hidden_size)")
print("Nombre de tokens patchs :", (224//16)**2, "+ 1 token CLS → total 197 tokens")

# ===============================
# 3️⃣ Classification avec le token CLS
# ===============================
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits
pred_class_idx = logits.argmax(-1).item()
pred_class_label = model.config.id2label[pred_class_idx]

print(f"\n✅ Classe prédite pour cat.jpg : {pred_class_label}")

# ===============================
# 4️⃣ Extraction et visualisation de l'attention du token CLS
# ===============================
with torch.no_grad():
    outputs_attn = model(**inputs, output_attentions=True)

# attentions : tuple de 12 éléments (couches), chaque tensor shape = (batch, n_heads, seq_len, seq_len)
attentions = outputs_attn.attentions
first_layer_attn = attentions[0]  # première couche
print("Shape de l'attention première couche :", first_layer_attn.shape)

# extraire l'attention du token CLS vers tous les patchs (index 0)
cls_attn = first_layer_attn[0, :, 0, 1:]  # shape = (n_heads, 196)
n_heads = cls_attn.shape[0]
print("Nombre de têtes :", n_heads)

# Visualiser attention de chaque tête
fig, axes = plt.subplots(1, n_heads, figsize=(15, 3))
for i in range(n_heads):
    attn_map = cls_attn[i].reshape(14, 14).detach().numpy()
    axes[i].imshow(attn_map, cmap='viridis')
    axes[i].set_title(f"Tête {i+1}")
    axes[i].axis('off')
plt.suptitle("Attention du token CLS → patchs (première couche)")
plt.show()

# ===============================
# 5️⃣ Comparaison et résumé
# ===============================
print("\nRésumé :")
print("- Taille de l'espace latent : 192")
print("- Image tokenisée en 196 patchs de 16x16 + token CLS")
print("- Premier token = token CLS → vecteur global représentant l'image")
print(f"- Classe prédite pour cat.jpg : {pred_class_label}")
print("- Attention du token CLS montre quelles zones de l'image influencent la décision")


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


FileNotFoundError: [Errno 2] No such file or directory: '../exam_session_anticipee/cat.jpg'